In [ ]:
import uuid
import pandas as pd
import psycopg2
from qdrant_client import QdrantClient
from qdrant_client.http import models
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from final_project.connect_to_google_drive import get_sheets_client, SHEET_ID

In [38]:
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

In [39]:
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

In [ ]:
sheets_client = get_sheets_client()
sheet = sheets_client.open_by_key(SHEET_ID).sheet1
df = pd.DataFrame(sheet.get_all_records())
to_sync = df[(df['status'] == 'Success') & (df['vector_db_sync'] == 'No')]

In [59]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [54]:
def get_text_from_postgres(file_id):
    try:
        conn = psycopg2.connect(
            dbname="ucu_rag_db", user="user", password="password", host="127.0.0.1", port=5432
        )
        cur = conn.cursor()
        cur.execute(
            "SELECT markdown_content FROM processed_documents WHERE google_drive_id = %s", 
            (file_id,)
        )
        result = cur.fetchone()
        cur.close()
        conn.close()
        return result[0] if result else None
    except Exception as e:
        print(f"ERROR: {e}")
        return None

In [ ]:
def upload_to_qdrant(to_sync, model, collection, chunking_method, e5=False):
    for _, row in to_sync.iterrows():
            file_id = row['google_drive_id']
            file_name = row['file_name']
            
            text = get_text_from_postgres(file_id)

            if text:
                if chunking_method == "fized_size":
                    chunks = [text[i:i+1000] for i in range(0, len(text), 800)]
                elif chunking_method == "recursive_character":
                    chunks = text_splitter.split_text(text)
                for i, chunk in enumerate(chunks):
                    if e5:
                        vector = model.encode("passage: " + chunk).tolist()
                    else:
                        vector = model.encode(chunk).tolist()
                    point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{file_id}_{i}"))
                    client.upsert(
                        collection_name=collection,
                        points=[
                            models.PointStruct(
                                id=point_id,
                                vector=vector,
                                payload={
                                    "file_id": file_id,
                                    "file_name": file_name,
                                    "text": chunk,
                                    "chunk_index": i
                                }
                            )
                        ]
                    )
                print(f"File {file_name} uploaded into Qdrant")

<hr>

# paraphrase-multilingual-MiniLM-L12-v2

In [ ]:
MODEL_BASELINE = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 236.69it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
COLLECTION_BASELINE = "ucu_documents_baseline"

In [ ]:
if not client.collection_exists(COLLECTION_BASELINE):
    client.create_collection(
        collection_name=COLLECTION_BASELINE,
        vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE)
    )

In [ ]:
upload_to_qdrant(to_sync=to_sync, model=MODEL_BASELINE, collection=COLLECTION_BASELINE)

File plagiarism_check_policy.pdf in progress
File academic_integrity_policy.pdf in progress
File Plan-rozvytku-polityky-rivnosti-ta-inklyuzyvnogo-robochogo-seredovyshhe-v-UKU.pdf in progress
File Polozhennya-pro-Otsinku-efektyvnosti-pratsivnykiv-Ukrayinskogo-katolytskogo-Universytetu-1.pdf in progress
File Polozhennya-pro-pidbir-pratsivnykiv-Ukrayinskogo-katolytskogo-universytetu-1.pdf in progress
File Pravyla-vnutrishnogo-trudovogo-rozporyadku-1.pdf in progress
File Polozhennya-pro-sluzhbovi-vidryadzhennya-personalu-ZVO-UKU.pdf in progress
File Polozhennya-pro-dystantsijnu-robotu-ta-gnuchkyj-rezhym-robochogo-chasu-Ukrayinskogo-katolytskogo-universytetu-1.pdf in progress
File Polozhennya-pro-shhorichnu-vidznaku-Vykladach-roku.pdf in progress
File Poryadok-realizatsiyi-programy-rozvytku-molodyh-naukovo-pedagogichnyh-ta-pedagogichnyh-pratsivnykiv-UKU.pdf in progress
File Polozhennya-pro-poryadok-vyznannya-inozemnyh-dokumentiv-pro-osvitu-v-UKU_2020.pdf in progress
File Zminy-do-Polozhenny

<hr>

# lang-uk/ukr-paraphrase-multilingual-mpnet-base

In [ ]:
MODEL_UKR = SentenceTransformer('lang-uk/ukr-paraphrase-multilingual-mpnet-base')

c:\Users\Olena\Desktop\ucu\four\rag_support_system\.venv\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Olena\.cache\huggingface\hub\models--lang-uk--ukr-paraphrase-multilingual-mpnet-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 292.52it/

In [19]:
COLLECTION_UKR = "ucu_documents_UKR"

In [24]:
if not client.collection_exists(COLLECTION_UKR):
    client.create_collection(
        collection_name=COLLECTION_UKR,
        vectors_config=models.VectorParams(size=768, distance=models.Distance.COSINE)
    )

In [27]:
upload_to_qdrant(to_sync=to_sync, model=MODEL_UKR, collection=COLLECTION_UKR)

File plagiarism_check_policy.pdf in progress
File academic_integrity_policy.pdf in progress
File Plan-rozvytku-polityky-rivnosti-ta-inklyuzyvnogo-robochogo-seredovyshhe-v-UKU.pdf in progress
File Polozhennya-pro-Otsinku-efektyvnosti-pratsivnykiv-Ukrayinskogo-katolytskogo-Universytetu-1.pdf in progress
File Polozhennya-pro-pidbir-pratsivnykiv-Ukrayinskogo-katolytskogo-universytetu-1.pdf in progress
File Pravyla-vnutrishnogo-trudovogo-rozporyadku-1.pdf in progress
File Polozhennya-pro-sluzhbovi-vidryadzhennya-personalu-ZVO-UKU.pdf in progress
File Polozhennya-pro-dystantsijnu-robotu-ta-gnuchkyj-rezhym-robochogo-chasu-Ukrayinskogo-katolytskogo-universytetu-1.pdf in progress
File Polozhennya-pro-shhorichnu-vidznaku-Vykladach-roku.pdf in progress
File Poryadok-realizatsiyi-programy-rozvytku-molodyh-naukovo-pedagogichnyh-ta-pedagogichnyh-pratsivnykiv-UKU.pdf in progress
File Polozhennya-pro-poryadok-vyznannya-inozemnyh-dokumentiv-pro-osvitu-v-UKU_2020.pdf in progress
File Zminy-do-Polozhenny

<hr>

# intfloat/multilingual-e5-small

In [56]:
MODEL_E5 = SentenceTransformer('intfloat/multilingual-e5-small')

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 173.53it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [57]:
COLLECTION_E5 = "ucu_documents_E5_v2"

In [58]:
if not client.collection_exists(COLLECTION_E5):
    client.create_collection(
        collection_name=COLLECTION_E5,
        vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE)
    )

In [ ]:
upload_to_qdrant(to_sync=to_sync, model=MODEL_E5, collection=COLLECTION_E5, e5=True)

File plagiarism_check_policy.pdf in progress
File academic_integrity_policy.pdf in progress
File Plan-rozvytku-polityky-rivnosti-ta-inklyuzyvnogo-robochogo-seredovyshhe-v-UKU.pdf in progress
File Polozhennya-pro-Otsinku-efektyvnosti-pratsivnykiv-Ukrayinskogo-katolytskogo-Universytetu-1.pdf in progress
File Polozhennya-pro-pidbir-pratsivnykiv-Ukrayinskogo-katolytskogo-universytetu-1.pdf in progress
File Pravyla-vnutrishnogo-trudovogo-rozporyadku-1.pdf in progress
File Polozhennya-pro-sluzhbovi-vidryadzhennya-personalu-ZVO-UKU.pdf in progress
File Polozhennya-pro-dystantsijnu-robotu-ta-gnuchkyj-rezhym-robochogo-chasu-Ukrayinskogo-katolytskogo-universytetu-1.pdf in progress
File Polozhennya-pro-shhorichnu-vidznaku-Vykladach-roku.pdf in progress
File Poryadok-realizatsiyi-programy-rozvytku-molodyh-naukovo-pedagogichnyh-ta-pedagogichnyh-pratsivnykiv-UKU.pdf in progress
File Polozhennya-pro-poryadok-vyznannya-inozemnyh-dokumentiv-pro-osvitu-v-UKU_2020.pdf in progress
File Zminy-do-Polozhenny

<hr>

# intfloat/multilingual-e5-base

In [25]:
MODEL_E5_BASE = SentenceTransformer('intfloat/multilingual-e5-base')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 306.10it/s, Materializing param=pooler.dense.weight]                               
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
COLLECTION_E5_BASE = "ucu_documents_E5_base_v2"

In [27]:
if not client.collection_exists(COLLECTION_E5_BASE):
    client.create_collection(
        collection_name=COLLECTION_E5_BASE,
        vectors_config=models.VectorParams(size=768, distance=models.Distance.COSINE)
    )

In [ ]:
upload_to_qdrant(to_sync, MODEL_E5_BASE, COLLECTION_E5_BASE, e5=True)

File plagiarism_check_policy.pdf in progress
File academic_integrity_policy.pdf in progress
File Plan-rozvytku-polityky-rivnosti-ta-inklyuzyvnogo-robochogo-seredovyshhe-v-UKU.pdf in progress
File Polozhennya-pro-Otsinku-efektyvnosti-pratsivnykiv-Ukrayinskogo-katolytskogo-Universytetu-1.pdf in progress
File Polozhennya-pro-pidbir-pratsivnykiv-Ukrayinskogo-katolytskogo-universytetu-1.pdf in progress
File Pravyla-vnutrishnogo-trudovogo-rozporyadku-1.pdf in progress
File Polozhennya-pro-sluzhbovi-vidryadzhennya-personalu-ZVO-UKU.pdf in progress
File Polozhennya-pro-dystantsijnu-robotu-ta-gnuchkyj-rezhym-robochogo-chasu-Ukrayinskogo-katolytskogo-universytetu-1.pdf in progress
File Polozhennya-pro-shhorichnu-vidznaku-Vykladach-roku.pdf in progress
File Poryadok-realizatsiyi-programy-rozvytku-molodyh-naukovo-pedagogichnyh-ta-pedagogichnyh-pratsivnykiv-UKU.pdf in progress
File Polozhennya-pro-poryadok-vyznannya-inozemnyh-dokumentiv-pro-osvitu-v-UKU_2020.pdf in progress
File Zminy-do-Polozhenny